# Classificação Estruturada em Português com Gemini 2.5 Flash

Este notebook está configurado para retornar apenas dados em **Português do Brasil**, utilizando o modelo **Gemini 2.5 Flash** e o SDK `google-genai`.

In [1]:
import os
import time
import json
import PIL.Image
from google import genai
from pydantic import BaseModel, Field
from typing import List, Optional
from dotenv import load_dotenv
from pathlib import Path
from IPython.display import display, JSON

## Definição do Esquema em Português

Utilizamos o `Field` do Pydantic para dar instruções explícitas ao modelo sobre o idioma de cada campo.

In [2]:
class Deteccao(BaseModel):
    categoria: str = Field(description="Categoria do animal em português (gato, cachorro, pássaro, etc)")
    raca: Optional[str] = Field(description="Raça identificada em português. Se não souber, retorne nulo.")
    descricao: str = Field(description="Descrição detalhada do que o animal está fazendo, exclusivamente em português.")

class AnaliseImagem(BaseModel):
    categoria_principal: str = Field(description="Categoria geral da imagem em português.")
    deteccoes: List[Deteccao] = Field(description="Lista de animais detectados.")
    resumo_informativo: str = Field(description="Resumo executivo da imagem, formatado em português do Brasil.")
    sentimento: str = Field(description="Sentimento ou atmosfera da imagem (ex: alegre, calmo, tenso, nostálgico). Escreva em português do Brasil.")
    cores_predominantes: List[str] = Field(description="Lista das cores mais presentes na imagem em português do Brasil.")
    elementos_secundarios: List[str] = Field(description="Outros objetos ou elementos importantes (árvores, móveis, gramado, asfalto, casa, etc.) além dos principais.")
    localizacao_provavel: str = Field(description="Local onde a foto provavelmente foi tirada (casa, rua, campo, clínica veterinária, etc) em português do Brasil.")

In [3]:
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
print("Cliente configurado. Modelo: Gemini 2.5 Flash")

Cliente configurado. Modelo: Gemini 2.5 Flash


## Processando Imagens (Output 100% em Português)

O prompt e o esquema forçam o uso exclusivo da língua portuguesa.

In [4]:
image_folder = Path("imagens")
output_folder = Path("resultados_json")
output_folder.mkdir(exist_ok=True)

image_extensions = [".jpg", ".jpeg", ".png", ".webp"]
files = [f for f in image_folder.iterdir() if f.suffix.lower() in image_extensions]

print(f"Iniciando análise de {len(files)} imagens...")

for i, img_path in enumerate(files):
    print(f"\n[{i+1}/{len(files)}] Processando: {img_path.name}")
    
    try:
        img = PIL.Image.open(img_path)
        
        # Chamada com Gemini 2.5 Flash e esquema em PT-BR
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=["Analise esta imagem detalhadamente. Forneça todos os textos estritamente em Português do Brasil.", img],
            config={
                'response_mime_type': 'application/json',
                'response_schema': AnaliseImagem,
            }
        )
        
        # Obtém o objeto Pydantic e converte para dicionário
        dados = response.parsed
        dados_dict = dados.model_dump()
        
        # 1. Visualizar o JSON na célula do notebook
        display(JSON(dados_dict))
        
        # 2. Salvar o arquivo JSON na pasta do projeto
        json_file_path = output_folder / f"{img_path.stem}.json"
        with open(json_file_path, "w", encoding="utf-8") as f:
            json.dump(dados_dict, f, ensure_ascii=False, indent=4)
        
        print(f"Principal: {dados.categoria_principal}")
        print(f"Resumo: {dados.resumo_informativo}")
        print(f"Sentimento: {dados.sentimento}")
        print(f"Localização: {dados.localizacao_provavel}")
        print(f"Salvo em: {json_file_path}")
            
        time.sleep(2) # Pausa maior para evitar 429
        
    except Exception as e:
        print(f"Erro ao processar {img_path.name}: {e}")

Iniciando análise de 7 imagens...

[1/7] Processando: cachorrofofo.jpeg


<IPython.core.display.JSON object>

Principal: Animais
Resumo: A imagem mostra um adorável filhote de Golden Retriever com pelagem dourada, sentado e sorrindo para a câmera em um fundo branco. O cachorro exibe uma expressão alegre e cativante.
Sentimento: Alegre e encantador
Localização: Estúdio fotográfico
Salvo em: resultados_json/cachorrofofo.json

[2/7] Processando: conheca-as-racas-de-cachorros-mais-inteligentes-do-mundo_0.jpg


<IPython.core.display.JSON object>

Principal: Cães
Resumo: A imagem apresenta dois cães, um Golden Retriever e um Pastor Alemão, sentados em um gramado verde, ambos com olhares atentos direcionados para cima. O Golden Retriever está à esquerda com um colar, enquanto o Pastor Alemão, à direita, tem a boca entreaberta, sugerindo antecipação.
Sentimento: Atento, calmo e expectante
Localização: Parque ou quintal gramado
Salvo em: resultados_json/conheca-as-racas-de-cachorros-mais-inteligentes-do-mundo_0.json

[3/7] Processando: images (3).jpeg
Erro ao processar images (3).jpeg: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 